# BioSignal Lab: units, provenance and subject grouping

This executable tutorial uses calibration tones and simulated subject identifiers only. It demonstrates software behavior, not physiological validity or clinical accuracy. Install the repository with `python -m pip install -e '.[all,notebooks]'`, then run this notebook using that environment.

In [1]:
import numpy as np
from biosignal_lab.engine import Signal, FilterSpec, filter_signal, welch_psd, integrate_band

fs = 250
t = np.arange(fs * 10) / fs
signal = Signal((10 * np.sin(2 * np.pi * 10 * t))[None], fs, ('Calibration',),
                unit='uV', reference='synthetic source', metadata={'seed': 42})
f, power = welch_psd(signal, window_s=2)
alpha_power = float(integrate_band(f, power, 8, 13)[0])
assert np.isclose(alpha_power, 50, rtol=0.01)
f_v, power_v = welch_psd(signal.to_unit('V'), window_s=2)
assert np.allclose(power_v, power * 1e-12, rtol=1e-10, atol=1e-25)
print(f'10 uV peak calibration tone: {alpha_power:.3f} uV^2 in 8-13 Hz')

10 uV peak calibration tone: 50.000 uV^2 in 8-13 Hz


## Filtering is a declared scientific choice

Forward/backward filtering is acausal and squares the single-pass magnitude response. Its edge padding and transient limitations remain part of the result. Shared SciPy routines do not constitute an independent numerical reference.

In [2]:
filtered = filter_signal(signal, FilterSpec(cutoff=(1, 40), phase='zero'))
assert signal.unit == filtered.unit == 'uV'
assert filtered.reference == signal.reference
assert filtered.history[-1]['operation'] == 'filter'
filtered.history[-1]

{'operation': 'filter',
 'parameters': {'family': 'butter',
  'kind': 'bandpass',
  'cutoff': (1, 40),
  'order': 4,
  'phase': 'zero',
  'ripple_db': 1.0,
  'attenuation_db': 40.0,
  'quality': 30.0,
  'padding': 'odd',
  'pad_samples': 27,
  'note': 'Zero phase is acausal and squares the single-pass magnitude response; edge transients remain.'}}

## Execute a typed experiment

The JSON definition is data. It names registered implementations; it cannot import arbitrary Python. The manifest binds source, workflow and node results. Trusted custom plugins are executable code and require separate review.

In [3]:
from biosignal_lab.studio import Workflow, Executor, builtin_registry
workflow = Workflow.from_dict({'schema_version': 1, 'name': 'Calibration workflow', 'nodes': [
    {'id': 'demo', 'kind': 'simulate', 'parameters': {'duration_s': 10, 'n_channels': 2}},
    {'id': 'analysis', 'kind': 'analyze', 'inputs': {'signal': {'node': 'demo'}}},
    {'id': 'summary', 'kind': 'summary', 'inputs': {'analysis': {'node': 'analysis'}}},
]})
execution = Executor(builtin_registry()).execute(workflow)
summary = execution.outputs['summary']['out']
assert summary['n_channels'] == 2
print('Executed nodes:', len(execution.manifest['nodes']))
print('Workflow SHA-256:', workflow.digest)

Executed nodes: 3
Workflow SHA-256: c07c3d75e333ba3cfa4d08e444c37ff3b117de7dce2bbe5c67ad74e68a6eff3e


## Hold out complete simulated subjects

The following deliberately separable tone task tests the evaluation machinery. Its accuracy is not evidence for motor imagery, P300, gesture recognition or any real physiological task. Scaling is fitted only on training folds; subject identifiers must represent real independence when using real datasets.

In [4]:
from biosignal_lab.ai.demo import synthetic_feature_dataset
from biosignal_lab.ai import evaluate_grouped

dataset = synthetic_feature_dataset(seed=42, n_subjects=4, trials_per_subject=8)
report = evaluate_grouped(dataset, model='lda', cv='loso', seed=42, bootstrap_samples=100)
for fold in report['folds']:
    assert not set(fold['train_subjects']) & set(fold['test_subjects'])
print('Synthetic subjects:', report['n_subjects'])
print('Synthetic tone accuracy:', report['metrics']['accuracy'])
print(report['uncertainty_note'])

Synthetic subjects: 4
Synthetic tone accuracy: 1.0
Percentile bootstrap of subject accuracies; small cohorts and overlapping training folds limit inference.


## Exercises

1. Change the tone to 20 Hz. Predict the 8–13 Hz power before running the calculation.
2. Compare causal and zero-phase filtering at both recording edges. Explain why they differ.
3. Convert microvolts to volts and explain the squared-unit PSD conversion.
4. Shuffle the synthetic labels and rerun grouped evaluation. Preserve the failed/negative control rather than selecting a favorable seed.
5. Explain why repeated windows from one participant must remain in one evaluation group.

See `docs/PLATFORM_GUIDE.md`, `docs/ENGINE.md`, `docs/AI_LAB.md` and `docs/PLATFORM_ACCEPTANCE.md` for implementation scope and incomplete external validation gates.